# PCA-MSPC end-to-end example

This notebook demonstrates a Ferrer-style latent-structure MSPC workflow:

1. build a correlated candidate Phase-I reference set;
2. curate it with the joint T²/SPE monitor;
3. monitor Phase-II observations; and
4. diagnose a correlation-breaking SPE event.

The synthetic data make the notebook runnable without a network connection or external files. Curation results must still be reviewed before declaring a real process reference set in control.

In [1]:
import numpy as np
import pandas as pd

from pca_tools import PCAOptimizer

ALPHA = 0.99
N_COMPONENTS = 3
rng = np.random.default_rng(2026)


def make_process_data(n_observations, rng):
    """Six correlated process variables generated from three latent factors."""
    latent_scores = rng.normal(size=(n_observations, N_COMPONENTS))
    loadings = np.array([
        [1.00, 0.20, 0.10], [0.85, -0.30, 0.15],
        [0.55, 0.75, -0.20], [-0.10, 0.80, 0.45],
        [0.25, -0.40, 0.95], [0.70, 0.10, 0.65],
    ])
    noise = rng.normal(scale=0.08, size=(n_observations, loadings.shape[0]))
    return pd.DataFrame(
        latent_scores @ loadings.T + noise,
        columns=["temperature", "pressure", "flow", "level", "vibration", "power"],
    )


def add_phase_i_contamination(data):
    contaminated = data.copy()
    contaminated.loc[5, ["temperature", "pressure", "flow"]] += 8.0
    contaminated.loc[17, "vibration"] += 12.0
    contaminated.loc[42, ["level", "power"]] -= 9.0
    return contaminated


def add_phase_ii_events(data):
    monitored = data.copy()
    monitored.loc[3, ["temperature", "pressure", "flow", "power"]] += 7.0
    monitored.loc[9, "vibration"] += 12.0
    return monitored


## 1. Create the candidate reference and monitoring data

The Phase-I candidate set contains mostly normal observations plus three deliberately introduced events. The Phase-II set is held out from fitting and includes a coordinated shift and an isolated variable excursion. In a real application, replace these synthetic frames with time-aligned process historian data after validating units, missing values, and operating-mode consistency.

In [2]:
candidate_reference = add_phase_i_contamination(make_process_data(300, rng))
phase_ii = add_phase_ii_events(make_process_data(80, rng))

# alpha is chart confidence; max_outlier_fraction is a separate curation policy.
optimizer = PCAOptimizer(
    n_comps=N_COMPONENTS,
    alpha=ALPHA,
    statistic="both",
    max_outlier_fraction=0.0,
    drop_percentage=0.25,
)
in_control = optimizer.optimize(candidate_reference)
audit = optimizer.result_
model = audit.model

print(f"Retained {len(in_control)} of {len(candidate_reference)} candidate reference rows")
print(audit.termination_reason)
pd.Series(model.control_limits_).round(3)

Retained 291 of 300 candidate reference rows
no observations exceed the selected Phase I limits


T2_phase1    12.622
T2_phase2    13.242
SPE           0.068
DModX         0.154
dtype: float64

## 2. Monitor Phase II observations

T² measures how unusual an observation is within the latent score space. SPE (also called Q) measures the residual distance from that space, so it is sensitive to changes in the learned correlation structure. A row is flagged when either statistic exceeds its Phase-II limit. Treat a flag as a prompt for investigation, not as proof that the measurement is bad.

In [3]:
t2, spe, residuals, scores = model.project(phase_ii)
monitoring = pd.DataFrame({"T2": t2, "SPE": spe}, index=phase_ii.index)
monitoring["T2_alarm"] = monitoring["T2"] > model.control_limits_["T2_phase2"]
monitoring["SPE_alarm"] = monitoring["SPE"] > model.control_limits_["SPE"]
monitoring["alarm"] = monitoring[["T2_alarm", "SPE_alarm"]].any(axis=1)
monitoring.loc[monitoring["alarm"]]

,T2,SPE,T2_alarm,SPE_alarm,alarm
1,1.623622,0.069852,False,True,True
3,116.808824,5.680909,True,True,True
9,56.559391,49.389525,True,True,True
77,13.256962,0.017741,True,False,True


## 3. Visualize the latent structure and control charts

The score plot overlays Phase-II observations on the Phase-I model. Use it to see clusters, shifts, and unusual trajectories; do not infer root cause from position alone. The next two charts show the Phase-II T² and SPE series against their respective control limits.

In [4]:
# Interactive Phase-I/II charts. In a notebook, leave each expression as
# the last item in the cell to render it.
model.biplot(1, 2, test_set=phase_ii)

alt.LayerChart(...)

### Hotelling's T² chart

An exceedance indicates an extreme combination of values that still broadly follows the latent correlation pattern. Compare the flagged timestamp with operating conditions, recipes, and upstream events.

In [5]:
model.hotelling_t2_plot_p2(phase_ii)

alt.LayerChart(...)

### SPE/Q chart

An SPE exceedance indicates that the observation is poorly reconstructed by the latent model. This is often associated with a changed correlation pattern, a sensor issue, or a new process event.

In [6]:
model.spe_plot_p2(phase_ii)

alt.LayerChart(...)

## 4. Diagnose a residual-space alarm

The contribution chart ranks variables by their contribution to the largest SPE alarm. It narrows the investigation; it does not establish causality. Confirm the result with engineering knowledge, instrument checks, and any relevant process sequence.

In [7]:
# Diagnose the largest residual-space event. The returned table should be
# reviewed alongside process context before acting on the alarm.
spe_alarm_index = monitoring["SPE"].idxmax()
contribution_chart, contributions = model.spe_contribution_plot(phase_ii.loc[[spe_alarm_index]])
display(contribution_chart)
contributions.head()

alt.Chart(...)

,variable,contribution,relative_contribution
0,vibration,19.088151,0.386482
1,flow,13.392391,0.271159
2,power,13.243307,0.268140
3,level,2.075535,0.042024
4,pressure,1.433854,0.029032


## 5. Review the Phase-I curation audit

`removed_data` contains the observations excluded during curation. `history` records the limits, flagged fraction, and removals at every iteration. Persist this audit with the approved model so that the construction of the Phase-I baseline remains reproducible and reviewable.

In [8]:
# Audit exactly what Phase-I curation removed and why.
audit.removed_data, audit.history

(     temperature  pressure      flow     level  vibration     power
 5       7.379466  7.658182  7.230948 -0.015066   0.472193 -0.098831
 16      3.397521  1.300103  3.478084  2.986260   0.948790  3.225590
 17      0.827044  1.106592 -0.335572 -0.130425  13.408325  1.152089
 42      0.514282 -0.393880  1.466249 -7.838820  -0.940006 -8.863882
 62      0.473472  0.315553  0.987148 -1.982251  -3.367726 -1.981133
 79      0.994226  1.015904  0.273069  1.374249   2.042775  1.887713
 138     1.022162  1.467356  0.155615 -0.282326   0.851419  0.981391
 221     1.158366  0.828137  0.494864 -0.455441   0.127248  0.826455
 225    -2.298544 -2.373987 -0.953474  0.356785  -0.776919 -1.678241,
 [OptimizationIteration(iteration=0, n_samples=300, n_flagged=4, flagged_fraction=0.013333333333333334, limits={'T2': 12.62799339443119, 'SPE': 13.898985065769304}, removed_positions=[42], removed_index=[42]),
  OptimizationIteration(iteration=1, n_samples=299, n_flagged=4, flagged_fraction=0.013377926421404